# Python 進階教學（互動版 Notebook）

本 Notebook 對應《python進階教學.md》，涵蓋物件導向（OOP）、檔案操作，以及專案結構解析。

執行環境請先完成 uv 虛擬環境設定（詳見《新生教學_開發環境與伺服器設定.md》）。

> **本篇範例只用 Python 標準函式庫，不需額外安裝套件，直接執行即可。**  
> Cell 之間有依賴關係時（例如繼承範例需要先定義父類別），請由上往下依序執行。

## 1. 物件導向 OOP

物件導向（OOP）把資料（屬性）與操作資料的行為（方法）封裝在一起，方便組織複雜程式、重複利用程式碼。

### 1.1 物件類別 class

類別（class）是物件的「模板」，定義好屬性和方法之後，就可以用它建立任意多個實例（instance）：

In [ ]:
class Animal:
    """所有動物的基礎類別"""

    def __init__(self, name: str, age: int):
        # __init__ 是「建構式」：每次用 Animal(...) 建立新物件時自動執行
        # self 代表「正在被建立的這個物件本身」
        # 方法第一個參數固定是 self，Python 呼叫時會自動把物件傳進來
        self.name = name
        self.age = age

    def speak(self) -> str:
        return "..."

    def __str__(self) -> str:
        # __str__ 是 dunder method：print() 時自動呼叫
        return f"{self.__class__.__name__}({self.name}, {self.age}歲)"


a = Animal("Buddy", 3)
print(a)          # Animal(Buddy, 3歲)
print(a.name)     # Buddy
print(a.speak())  # ...

# 呼叫 a.speak() 等同 Animal.speak(a)，self 就是 a
print(Animal.speak(a))  # 效果相同

**`self`**：呼叫 `a.speak()` 時，Python 背後執行的是 `Animal.speak(a)`，把 `a` 自動傳入成為 `self`。`self` 不是關鍵字，只是慣例命名。

**`__init__`**：建構式，`Animal(...)` 建立新物件時自動執行，負責初始化屬性。

**`__str__`**：dunder method 之一，`print()` 呼叫時自動觸發，回傳物件的「字串表示」。

---

**`__name__` 進入點慣例**（在 `.py` 腳本中使用，在 Notebook 裡 `__name__` 固定是 `"__main__"`）：

In [ ]:
# 在 Notebook 中 __name__ 永遠是 "__main__"
# 在 .py 腳本裡，直接執行時也是 "__main__"，被 import 時才會是模組名稱
print(__name__)

# 實際腳本會這樣寫，讓這段只在「直接執行」時跑，被 import 時不跑：
if __name__ == "__main__":
    a = Animal("Rex", 5)
    print(a)

### 1.2 物件繼承 inheritance

子類別繼承父類別的屬性與方法，並可**覆寫（override）**方法做出不同行為，這就是多型（polymorphism）。

> 需先執行上方 1.1 的 Animal 定義 Cell。

In [ ]:
class Dog(Animal):
    def __init__(self, name: str, age: int, breed: str):
        super().__init__(name, age)  # 呼叫父類別 Animal 的 __init__
        self.breed = breed           # Dog 自己額外的屬性

    def speak(self) -> str:          # 覆寫父類別的 speak
        return "Woof!"


class Cat(Animal):                   # 沒覆寫 __init__，直接沿用父類別的
    def speak(self) -> str:
        return "Meow!"


animals = [Dog("Rex", 3, "Labrador"), Cat("Mimi", 2)]
for a in animals:
    print(a, "->", a.speak())
# Dog(Rex, 3歲) -> Woof!
# Cat(Mimi, 2歲) -> Meow!

print(isinstance(animals[0], Animal))  # True，Dog 是 Animal 的子類別
print(isinstance(animals[0], Dog))     # True
print(isinstance(animals[0], Cat))     # False

`super().__init__(name, age)` 的作用：把 `name`、`age` 的初始化邏輯交給父類別，避免重複寫。如果不呼叫，`self.name` 和 `self.age` 就不會被設定。

> `Cat` 沒有定義自己的 `__init__`，所以建立 `Cat("Mimi", 2)` 時直接沿用父類別 `Animal` 的 `__init__`，不需要 `super()`。只有**覆寫了 `__init__`** 時，才需要在裡面呼叫 `super().__init__()`。

### 1.3 產生器 generator

一般函式執行完一次回傳所有結果；**產生器**用 `yield` 關鍵字，每次呼叫只計算並回傳一個值，其餘時間暫停等待。先用記憶體用量來感受差異：

In [ ]:
import sys

big_list = [x ** 2 for x in range(1_000_000)]        # 一次全算完
big_gen  = (x ** 2 for x in range(1_000_000))        # 產生器運算式，幾乎不佔記憶體

print(f"list    記憶體：{sys.getsizeof(big_list):>12,} bytes")
print(f"generator 記憶體：{sys.getsizeof(big_gen):>10,} bytes")
# list 約 8 MB，generator 只有幾百 bytes

In [ ]:
# 產生器函式：用 yield 取代 return
def squares(n):
    for x in range(n):
        yield x ** 2    # 執行到這行就暫停、回傳值，下次 next() 才繼續

gen = squares(5)
print(next(gen))   # 0
print(next(gen))   # 1
print(next(gen))   # 4

print("--- for 迴圈走訪 ---")
for val in squares(5):
    print(val, end=" ")   # 0 1 4 9 16

In [ ]:
# 產生器運算式：() 而非 []
gen_expr = (x ** 2 for x in range(10))
print(type(gen_expr))          # <class 'generator'>
print(list(gen_expr))          # [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

# 走訪過一次就耗盡了，再次 list() 會是空的
print(list(gen_expr))          # []

### 1.4 裝飾器 decorator

裝飾器讓你在**不修改原函式程式碼**的情況下，把它「包裝」起來加上額外行為。語法是在函式定義上方寫 `@裝飾器名稱`：

In [ ]:
def shout(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@shout
def greet(name):
    return f"hello {name}"

print(greet("jack"))   # HELLO JACK

# @shout 等同於在定義後執行：greet = shout(greet)
# 以下兩行效果完全一樣：
def greet2(name):
    return f"hello {name}"
greet2 = shout(greet2)
print(greet2("jack"))  # HELLO JACK

In [ ]:
# 類別中常用的兩個內建裝飾器
class Counter:
    total_created = 0           # 類別屬性：所有實例共用同一份

    def __init__(self):
        Counter.total_created += 1

    def instance_info(self):
        return f"這是第 {Counter.total_created} 個實例"  # 一般方法，存取 self

    @classmethod
    def get_total(cls):          # cls 自動帶入「類別本身」
        return cls.total_created

    @staticmethod
    def description():           # 不需要 self / cls
        return "計算建立了多少個 Counter 實例"


c1 = Counter()
c2 = Counter()
c3 = Counter()

print(Counter.get_total())    # 3
print(Counter.description())  # 計算建立了多少個 Counter 實例
print(c1.instance_info())     # 這是第 3 個實例（total_created 已是 3）

簡單判斷原則：需要 `self.xxx`（某個物件的資料）→ 一般方法；需要 `cls.xxx`（整個類別的資料）→ `@classmethod`；完全不需要 `self`/`cls` → `@staticmethod`。

## 2. 檔案與路徑操作

### 2.1 傳統 open() 寫法

Python 內建的 `open()` 函式搭配 `with` 語法使用，可以確保檔案用完後自動關閉：

In [ ]:
# 寫入
with open("notes.txt", "w", encoding="utf-8") as f:
    f.write("第一行\n第二行\n第三行\n")

# 讀取全部
with open("notes.txt", "r", encoding="utf-8") as f:
    content = f.read()
print(content)

# 逐行讀取（大檔案時更省記憶體）
with open("notes.txt", "r", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        print(f"  第 {i} 行：{line.rstrip()}")  # rstrip() 去除行尾換行符

### 2.2 現代寫法：pathlib

`pathlib` 用物件導向的方式處理路徑，比字串拼接（`os.path.join`）更直覺、跨平台：

In [ ]:
from pathlib import Path

# 建立暫存資料夾
base = Path("temp_pathlib_demo")
base.mkdir(exist_ok=True)

print("是否存在:", base.exists())
print("是否為資料夾:", base.is_dir())

# 用 / 運算子組合路徑，寫入並讀取文字檔
log_file = base / "log.txt"
log_file.write_text("第一行\n第二行\n", encoding="utf-8")
print(log_file.read_text(encoding="utf-8"))

# 路徑資訊
print("副檔名:", log_file.suffix)           # .txt
print("檔名(不含副檔名):", log_file.stem)   # log
print("上層資料夾:", log_file.parent)

# 建立子資料夾並列出
sub = base / "subdir"
sub.mkdir(exist_ok=True)
(sub / "a.py").write_text("# a", encoding="utf-8")
(sub / "b.py").write_text("# b", encoding="utf-8")

print("\n遞迴找所有 .py:")
for p in base.rglob("*.py"):
    print(" ", p.relative_to(base))

In [ ]:
# 清理暫存資料夾（執行完可選擇性執行）
import shutil
shutil.rmtree(base)
print("temp_pathlib_demo 已刪除:", not base.exists())